# Memory and Conversation

## What you'll learn
- Why a model forgets everything between calls
- How to give an agent memory by keeping the conversation history
- How follow-up questions ("and Europe?") rely on that history
- The cost of a long history, and when to reset it

In [ ]:
import json
import pandas as pd

df = pd.read_csv("data/fpa_actuals_budget.csv")

# Derived lines every FP&A pack uses (all amounts in USD thousands).
df["Gross_Profit"] = df["Revenue"] - df["COGS"]
df["Budget_Gross_Profit"] = df["Budget_Revenue"] - df["Budget_COGS"]
df["Operating_Income"] = df["Gross_Profit"] - df["OpEx"]
df["Budget_Operating_Income"] = df["Budget_Gross_Profit"] - df["Budget_OpEx"]

print(df.shape)
df.head()

In [ ]:
import os
import importlib
from getpass import getpass

# Choose your provider: "anthropic" (Claude) or "openai" (ChatGPT).
PROVIDER = "anthropic"

MODELS = {"anthropic": "claude-opus-5", "openai": "gpt-5.6"}
KEY_NAMES = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY"}
MODEL = MODELS[PROVIDER]
sdk = importlib.import_module(PROVIDER)   # the anthropic or openai package


def make_client():
    """Create an API client, or return None (with a message) if there is no key."""
    key_name = KEY_NAMES[PROVIDER]
    if not os.environ.get(key_name):
        print(f"{key_name} is not set. Paste your key below, or press Enter to skip.")
        print("(The key stays in memory only. See 02_api_key_and_first_call.ipynb.)")
        try:
            key = getpass(f"{key_name}: ").strip()
        except Exception:
            key = ""
        if not key:
            print("No key, so the model cells in this notebook will be skipped.")
            return None
        os.environ[key_name] = key
    if PROVIDER == "anthropic":
        return sdk.Anthropic()
    return sdk.OpenAI()


client = make_client()
print("Provider:", PROVIDER, "| Model:", MODEL, "| Ready:", client is not None)

In [ ]:
def explain_error(error):
    """Turn an API error into a plain message."""
    if isinstance(error, sdk.AuthenticationError):
        return "The API key was rejected. Check that it is correct and still active."
    if isinstance(error, sdk.RateLimitError):
        return "Rate limit or usage quota reached. Wait a minute, or check your account billing."
    if isinstance(error, sdk.APIConnectionError):
        return "Could not reach the API. Check your internet connection or company proxy."
    if isinstance(error, sdk.APIStatusError):
        return f"The API returned an error ({error.status_code}): {error.message}"
    return f"Unexpected API error: {error}"

In [ ]:
METRICS = ["Revenue", "COGS", "Gross_Profit", "OpEx", "Operating_Income"]
PRODUCT_LINES = sorted(df["ProductLine"].unique())
REGIONS = sorted(df["Region"].unique())
FIRST_MONTH, LAST_MONTH = df["Month"].min(), df["Month"].max()


def select_rows(month_from, month_to, product_line=None, region=None):
    """Filter the data, raising ValueError with a clear message for bad inputs."""
    for month in (month_from, month_to):
        if len(month) != 7 or not FIRST_MONTH <= month <= LAST_MONTH:
            raise ValueError(f"month '{month}' must be YYYY-MM between {FIRST_MONTH} and {LAST_MONTH}")
    if month_from > month_to:
        raise ValueError("month_from must be on or before month_to")
    if product_line is not None and product_line not in PRODUCT_LINES:
        raise ValueError(f"unknown product_line '{product_line}'. Choose from {PRODUCT_LINES}")
    if region is not None and region not in REGIONS:
        raise ValueError(f"unknown region '{region}'. Choose from {REGIONS}")

    rows = df[(df["Month"] >= month_from) & (df["Month"] <= month_to)]
    if product_line is not None:
        rows = rows[rows["ProductLine"] == product_line]
    if region is not None:
        rows = rows[rows["Region"] == region]
    return rows


def compare(actual, budget):
    """Actual vs budget with the variance in money and percent."""
    return {
        "actual": round(float(actual), 1),
        "budget": round(float(budget), 1),
        "variance": round(float(actual - budget), 1),
        "variance_pct": round(float((actual - budget) / budget * 100), 1),
    }


def get_pnl_summary(month_from, month_to, product_line=None, region=None):
    """Actual vs budget for every P&L line, plus gross margin percent."""
    rows = select_rows(month_from, month_to, product_line, region)
    summary = {m: compare(rows[m].sum(), rows[f"Budget_{m}"].sum()) for m in METRICS}
    summary["Gross_Margin_pct"] = {
        "actual": round(float(rows["Gross_Profit"].sum() / rows["Revenue"].sum() * 100), 1),
        "budget": round(float(rows["Budget_Gross_Profit"].sum() / rows["Budget_Revenue"].sum() * 100), 1),
    }
    return summary


def get_variance_by(metric, month_from, month_to, by):
    """Variance of one metric split by ProductLine or Region, biggest variance first."""
    if metric not in METRICS:
        raise ValueError(f"unknown metric '{metric}'. Choose from {METRICS}")
    if by not in ("ProductLine", "Region"):
        raise ValueError("by must be 'ProductLine' or 'Region'")
    rows = select_rows(month_from, month_to)
    grouped = rows.groupby(by)[[metric, f"Budget_{metric}"]].sum()
    result = {name: compare(r[metric], r[f"Budget_{metric}"]) for name, r in grouped.iterrows()}
    return dict(sorted(result.items(), key=lambda item: -abs(item[1]["variance"])))


def get_monthly_trend(metric, month_from, month_to, product_line=None, region=None):
    """Month-by-month actual vs budget for one metric, with the average FX rate."""
    if metric not in METRICS:
        raise ValueError(f"unknown metric '{metric}'. Choose from {METRICS}")
    rows = select_rows(month_from, month_to, product_line, region)
    grouped = rows.groupby("Month").agg(
        actual=(metric, "sum"), budget=(f"Budget_{metric}", "sum"), fx_rate=("FX_Rate", "mean"))
    return {
        month: {**compare(r["actual"], r["budget"]), "fx_rate": round(float(r["fx_rate"]), 3)}
        for month, r in grouped.iterrows()
    }


print(get_pnl_summary("2025-04", "2025-06")["Revenue"])

In [ ]:
MONTH = {"type": "string", "description": "Month in YYYY-MM format"}
METRIC = {"type": "string", "enum": METRICS, "description": "P&L line"}
PRODUCT = {"type": "string", "enum": PRODUCT_LINES, "description": "Optional product line filter"}
REGION = {"type": "string", "enum": REGIONS, "description": "Optional region filter"}

TOOLS = [
    {
        "name": "get_pnl_summary",
        "description": "Actual vs budget for every P&L line (USD thousands) over a month range, "
                       "optionally for one product line and/or region. Includes gross margin %.",
        "parameters": {
            "type": "object",
            "properties": {"month_from": MONTH, "month_to": MONTH,
                           "product_line": PRODUCT, "region": REGION},
            "required": ["month_from", "month_to"],
            "additionalProperties": False,
        },
    },
    {
        "name": "get_variance_by",
        "description": "Variance to budget for one P&L line, split by ProductLine or Region, "
                       "biggest variance first. Use it to find what drove a variance.",
        "parameters": {
            "type": "object",
            "properties": {"metric": METRIC, "month_from": MONTH, "month_to": MONTH,
                           "by": {"type": "string", "enum": ["ProductLine", "Region"]}},
            "required": ["metric", "month_from", "month_to", "by"],
            "additionalProperties": False,
        },
    },
    {
        "name": "get_monthly_trend",
        "description": "Month-by-month actual vs budget for one P&L line, with the average FX "
                       "rate (1.0 = budget rate; below 1.0 means a weaker local currency).",
        "parameters": {
            "type": "object",
            "properties": {"metric": METRIC, "month_from": MONTH, "month_to": MONTH,
                           "product_line": PRODUCT, "region": REGION},
            "required": ["metric", "month_from", "month_to"],
            "additionalProperties": False,
        },
    },
]

FUNCTIONS = {
    "get_pnl_summary": get_pnl_summary,
    "get_variance_by": get_variance_by,
    "get_monthly_trend": get_monthly_trend,
}


def run_tool(name, arguments):
    """Run one tool call. Returns (text for the model, is_error)."""
    if name not in FUNCTIONS:
        return f"Error: unknown tool '{name}'", True
    try:
        return json.dumps(FUNCTIONS[name](**arguments)), False
    except (TypeError, ValueError) as error:
        return f"Error: {error}", True


def provider_tools():
    """Convert TOOLS into the format the chosen provider expects."""
    if PROVIDER == "anthropic":
        return [{"name": t["name"], "description": t["description"],
                 "input_schema": t["parameters"]} for t in TOOLS]
    return [{"type": "function", "name": t["name"], "description": t["description"],
             "parameters": t["parameters"]} for t in TOOLS]

In [ ]:
SYSTEM_PROMPT = """You are an FP&A analyst assistant for a fictional company.
All amounts are in USD thousands. Months are YYYY-MM, from 2023-01 to 2025-12.
Always get numbers from the tools; never guess or invent figures.
For cost lines (COGS, OpEx) a positive variance (actual above budget) is unfavourable.
Answer in a few short sentences or bullets and name the period you looked at."""

## Models have no memory

Each API call is independent. The model does not remember your last question unless you
send it again. The `run_agent` function from lesson 5 started a fresh list of messages on
every call, so a follow-up like "and what about Europe?" would make no sense to it.

**Memory is just a list you keep and resend.** The `Conversation` class below stores the
full history (questions, tool calls, tool results, answers) and sends all of it with each
new question.

In [ ]:
class Conversation:
    """An agent that remembers earlier questions and answers."""

    def __init__(self, system=SYSTEM_PROMPT, max_steps=8):
        self.system = system
        self.max_steps = max_steps
        self.history = []   # everything said so far, in the provider's own format

    def ask(self, question, verbose=True):
        if client is None:
            return "(Skipped: no API key. See 02_api_key_and_first_call.ipynb.)"
        self.history.append({"role": "user", "content": question})
        tools = provider_tools()
        try:
            for step in range(self.max_steps):
                if PROVIDER == "anthropic":
                    response = client.messages.create(
                        model=MODEL, max_tokens=16000, system=self.system,
                        tools=tools, messages=self.history)
                    self.history.append({"role": "assistant", "content": response.content})
                    calls = [b for b in response.content if b.type == "tool_use"]
                    if not calls:
                        return "".join(b.text for b in response.content if b.type == "text")
                    results = []
                    for call in calls:
                        if verbose:
                            print(f"step {step + 1}: {call.name}({call.input})")
                        output, is_error = run_tool(call.name, call.input)
                        results.append({"type": "tool_result", "tool_use_id": call.id,
                                        "content": output, "is_error": is_error})
                    self.history.append({"role": "user", "content": results})
                else:
                    response = client.responses.create(
                        model=MODEL, instructions=self.system, tools=tools, input=self.history)
                    self.history += response.output
                    calls = [item for item in response.output if item.type == "function_call"]
                    if not calls:
                        return response.output_text
                    for call in calls:
                        arguments = json.loads(call.arguments)
                        if verbose:
                            print(f"step {step + 1}: {call.name}({arguments})")
                        output, is_error = run_tool(call.name, arguments)
                        self.history.append({"type": "function_call_output",
                                             "call_id": call.call_id, "output": output})
        except sdk.APIError as error:
            print(explain_error(error))
            return None
        return f"Stopped after {self.max_steps} steps without a final answer."

    def reset(self):
        self.history = []

In [ ]:
chat = Conversation()

print(chat.ask("What was total revenue vs budget for 2025-04 to 2025-06?"))

In [ ]:
# A follow-up that only makes sense with memory.
print(chat.ask("Which region drove most of that variance?"))

In [ ]:
print(chat.ask("Was that caused by volume or by currency? Check the FX rate."))

## What the model now sees

Every question adds several entries: your question, the model's tool calls, the tool
results, and the answer. All of it is sent on the next call.

In [ ]:
print("Entries in history:", len(chat.history))

## The cost of memory

A longer history means more tokens on every call, so each follow-up costs a little more
and takes a little longer. Practical rules:

- Start a new `Conversation` (or call `chat.reset()`) when the topic changes.
- Keep tool results small (these tools return summaries, not raw rows).
- For very long sessions, providers offer features that summarise or trim old history;
  see their documentation when you need them.

In [ ]:
chat.reset()
print("Entries after reset:", len(chat.history))

## Practice

1. Start a new conversation. Ask about Q4 2024 revenue, then follow up with "Which product
   line?" and then "Which region within that product line?".
2. Ask a follow-up **after** calling `reset()`. What happens?
3. Print `len(chat.history)` after each question in exercise 1. How fast does it grow?
4. Add a method `last_answer_only()` that returns a new `Conversation` whose history holds
   only the last question and answer as plain text (a simple way to trim memory).

In [ ]:
# Your practice code here

## Summary

- The model forgets everything between calls; memory is the history you resend.
- Keep one history per conversation and append questions, tool calls, results, and answers.
- Follow-up questions work because the earlier turns are in that history.
- Longer history costs more; reset when the topic changes and keep tool results small.

## Practice Solutions

In [ ]:
import json
import pandas as pd

df = pd.read_csv("data/fpa_actuals_budget.csv")

# Derived lines every FP&A pack uses (all amounts in USD thousands).
df["Gross_Profit"] = df["Revenue"] - df["COGS"]
df["Budget_Gross_Profit"] = df["Budget_Revenue"] - df["Budget_COGS"]
df["Operating_Income"] = df["Gross_Profit"] - df["OpEx"]
df["Budget_Operating_Income"] = df["Budget_Gross_Profit"] - df["Budget_OpEx"]




import os
import importlib
from getpass import getpass

# Choose your provider: "anthropic" (Claude) or "openai" (ChatGPT).
PROVIDER = "anthropic"

MODELS = {"anthropic": "claude-opus-5", "openai": "gpt-5.6"}
KEY_NAMES = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY"}
MODEL = MODELS[PROVIDER]
sdk = importlib.import_module(PROVIDER)   # the anthropic or openai package


def make_client():
    """Create an API client, or return None (with a message) if there is no key."""
    key_name = KEY_NAMES[PROVIDER]
    if not os.environ.get(key_name):
        print(f"{key_name} is not set. Paste your key below, or press Enter to skip.")
        print("(The key stays in memory only. See 02_api_key_and_first_call.ipynb.)")
        try:
            key = getpass(f"{key_name}: ").strip()
        except Exception:
            key = ""
        if not key:
            print("No key, so the model cells in this notebook will be skipped.")
            return None
        os.environ[key_name] = key
    if PROVIDER == "anthropic":
        return sdk.Anthropic()
    return sdk.OpenAI()


client = make_client()
print("Provider:", PROVIDER, "| Model:", MODEL, "| Ready:", client is not None)



def explain_error(error):
    """Turn an API error into a plain message."""
    if isinstance(error, sdk.AuthenticationError):
        return "The API key was rejected. Check that it is correct and still active."
    if isinstance(error, sdk.RateLimitError):
        return "Rate limit or usage quota reached. Wait a minute, or check your account billing."
    if isinstance(error, sdk.APIConnectionError):
        return "Could not reach the API. Check your internet connection or company proxy."
    if isinstance(error, sdk.APIStatusError):
        return f"The API returned an error ({error.status_code}): {error.message}"
    return f"Unexpected API error: {error}"



METRICS = ["Revenue", "COGS", "Gross_Profit", "OpEx", "Operating_Income"]
PRODUCT_LINES = sorted(df["ProductLine"].unique())
REGIONS = sorted(df["Region"].unique())
FIRST_MONTH, LAST_MONTH = df["Month"].min(), df["Month"].max()


def select_rows(month_from, month_to, product_line=None, region=None):
    """Filter the data, raising ValueError with a clear message for bad inputs."""
    for month in (month_from, month_to):
        if len(month) != 7 or not FIRST_MONTH <= month <= LAST_MONTH:
            raise ValueError(f"month '{month}' must be YYYY-MM between {FIRST_MONTH} and {LAST_MONTH}")
    if month_from > month_to:
        raise ValueError("month_from must be on or before month_to")
    if product_line is not None and product_line not in PRODUCT_LINES:
        raise ValueError(f"unknown product_line '{product_line}'. Choose from {PRODUCT_LINES}")
    if region is not None and region not in REGIONS:
        raise ValueError(f"unknown region '{region}'. Choose from {REGIONS}")

    rows = df[(df["Month"] >= month_from) & (df["Month"] <= month_to)]
    if product_line is not None:
        rows = rows[rows["ProductLine"] == product_line]
    if region is not None:
        rows = rows[rows["Region"] == region]
    return rows


def compare(actual, budget):
    """Actual vs budget with the variance in money and percent."""
    return {
        "actual": round(float(actual), 1),
        "budget": round(float(budget), 1),
        "variance": round(float(actual - budget), 1),
        "variance_pct": round(float((actual - budget) / budget * 100), 1),
    }


def get_pnl_summary(month_from, month_to, product_line=None, region=None):
    """Actual vs budget for every P&L line, plus gross margin percent."""
    rows = select_rows(month_from, month_to, product_line, region)
    summary = {m: compare(rows[m].sum(), rows[f"Budget_{m}"].sum()) for m in METRICS}
    summary["Gross_Margin_pct"] = {
        "actual": round(float(rows["Gross_Profit"].sum() / rows["Revenue"].sum() * 100), 1),
        "budget": round(float(rows["Budget_Gross_Profit"].sum() / rows["Budget_Revenue"].sum() * 100), 1),
    }
    return summary


def get_variance_by(metric, month_from, month_to, by):
    """Variance of one metric split by ProductLine or Region, biggest variance first."""
    if metric not in METRICS:
        raise ValueError(f"unknown metric '{metric}'. Choose from {METRICS}")
    if by not in ("ProductLine", "Region"):
        raise ValueError("by must be 'ProductLine' or 'Region'")
    rows = select_rows(month_from, month_to)
    grouped = rows.groupby(by)[[metric, f"Budget_{metric}"]].sum()
    result = {name: compare(r[metric], r[f"Budget_{metric}"]) for name, r in grouped.iterrows()}
    return dict(sorted(result.items(), key=lambda item: -abs(item[1]["variance"])))


def get_monthly_trend(metric, month_from, month_to, product_line=None, region=None):
    """Month-by-month actual vs budget for one metric, with the average FX rate."""
    if metric not in METRICS:
        raise ValueError(f"unknown metric '{metric}'. Choose from {METRICS}")
    rows = select_rows(month_from, month_to, product_line, region)
    grouped = rows.groupby("Month").agg(
        actual=(metric, "sum"), budget=(f"Budget_{metric}", "sum"), fx_rate=("FX_Rate", "mean"))
    return {
        month: {**compare(r["actual"], r["budget"]), "fx_rate": round(float(r["fx_rate"]), 3)}
        for month, r in grouped.iterrows()
    }






MONTH = {"type": "string", "description": "Month in YYYY-MM format"}
METRIC = {"type": "string", "enum": METRICS, "description": "P&L line"}
PRODUCT = {"type": "string", "enum": PRODUCT_LINES, "description": "Optional product line filter"}
REGION = {"type": "string", "enum": REGIONS, "description": "Optional region filter"}

TOOLS = [
    {
        "name": "get_pnl_summary",
        "description": "Actual vs budget for every P&L line (USD thousands) over a month range, "
                       "optionally for one product line and/or region. Includes gross margin %.",
        "parameters": {
            "type": "object",
            "properties": {"month_from": MONTH, "month_to": MONTH,
                           "product_line": PRODUCT, "region": REGION},
            "required": ["month_from", "month_to"],
            "additionalProperties": False,
        },
    },
    {
        "name": "get_variance_by",
        "description": "Variance to budget for one P&L line, split by ProductLine or Region, "
                       "biggest variance first. Use it to find what drove a variance.",
        "parameters": {
            "type": "object",
            "properties": {"metric": METRIC, "month_from": MONTH, "month_to": MONTH,
                           "by": {"type": "string", "enum": ["ProductLine", "Region"]}},
            "required": ["metric", "month_from", "month_to", "by"],
            "additionalProperties": False,
        },
    },
    {
        "name": "get_monthly_trend",
        "description": "Month-by-month actual vs budget for one P&L line, with the average FX "
                       "rate (1.0 = budget rate; below 1.0 means a weaker local currency).",
        "parameters": {
            "type": "object",
            "properties": {"metric": METRIC, "month_from": MONTH, "month_to": MONTH,
                           "product_line": PRODUCT, "region": REGION},
            "required": ["metric", "month_from", "month_to"],
            "additionalProperties": False,
        },
    },
]

FUNCTIONS = {
    "get_pnl_summary": get_pnl_summary,
    "get_variance_by": get_variance_by,
    "get_monthly_trend": get_monthly_trend,
}


def run_tool(name, arguments):
    """Run one tool call. Returns (text for the model, is_error)."""
    if name not in FUNCTIONS:
        return f"Error: unknown tool '{name}'", True
    try:
        return json.dumps(FUNCTIONS[name](**arguments)), False
    except (TypeError, ValueError) as error:
        return f"Error: {error}", True


def provider_tools():
    """Convert TOOLS into the format the chosen provider expects."""
    if PROVIDER == "anthropic":
        return [{"name": t["name"], "description": t["description"],
                 "input_schema": t["parameters"]} for t in TOOLS]
    return [{"type": "function", "name": t["name"], "description": t["description"],
             "parameters": t["parameters"]} for t in TOOLS]



SYSTEM_PROMPT = """You are an FP&A analyst assistant for a fictional company.
All amounts are in USD thousands. Months are YYYY-MM, from 2023-01 to 2025-12.
Always get numbers from the tools; never guess or invent figures.
For cost lines (COGS, OpEx) a positive variance (actual above budget) is unfavourable.
Answer in a few short sentences or bullets and name the period you looked at."""



class Conversation:
    """An agent that remembers earlier questions and answers."""

    def __init__(self, system=SYSTEM_PROMPT, max_steps=8):
        self.system = system
        self.max_steps = max_steps
        self.history = []   # everything said so far, in the provider's own format

    def ask(self, question, verbose=True):
        if client is None:
            return "(Skipped: no API key. See 02_api_key_and_first_call.ipynb.)"
        self.history.append({"role": "user", "content": question})
        tools = provider_tools()
        try:
            for step in range(self.max_steps):
                if PROVIDER == "anthropic":
                    response = client.messages.create(
                        model=MODEL, max_tokens=16000, system=self.system,
                        tools=tools, messages=self.history)
                    self.history.append({"role": "assistant", "content": response.content})
                    calls = [b for b in response.content if b.type == "tool_use"]
                    if not calls:
                        return "".join(b.text for b in response.content if b.type == "text")
                    results = []
                    for call in calls:
                        if verbose:
                            print(f"step {step + 1}: {call.name}({call.input})")
                        output, is_error = run_tool(call.name, call.input)
                        results.append({"type": "tool_result", "tool_use_id": call.id,
                                        "content": output, "is_error": is_error})
                    self.history.append({"role": "user", "content": results})
                else:
                    response = client.responses.create(
                        model=MODEL, instructions=self.system, tools=tools, input=self.history)
                    self.history += response.output
                    calls = [item for item in response.output if item.type == "function_call"]
                    if not calls:
                        return response.output_text
                    for call in calls:
                        arguments = json.loads(call.arguments)
                        if verbose:
                            print(f"step {step + 1}: {call.name}({arguments})")
                        output, is_error = run_tool(call.name, arguments)
                        self.history.append({"type": "function_call_output",
                                             "call_id": call.call_id, "output": output})
        except sdk.APIError as error:
            print(explain_error(error))
            return None
        return f"Stopped after {self.max_steps} steps without a final answer."

    def reset(self):
        self.history = []


# 1 and 3.
chat = Conversation()
for question in ["How did revenue compare to budget in 2024-10 to 2024-12?",
                 "Which product line?",
                 "Which region within that product line?"]:
    print(chat.ask(question, verbose=False))
    print("history entries:", len(chat.history), "\n")

# 2. After reset the model has no context, so "Which product line?" is unclear to it.
chat.reset()
print(chat.ask("Which product line?", verbose=False))


# 4.
def last_answer_only(conversation, question, answer):
    """A fresh conversation that remembers only one question and its answer."""
    trimmed = Conversation(system=conversation.system, max_steps=conversation.max_steps)
    trimmed.history = [{"role": "user", "content": question},
                       {"role": "assistant", "content": answer}]
    return trimmed